In [2]:
import qutip as qt
import numpy as np

# Define the Number of States in Fock Basis
N = 128
a = qt.destroy(N) # Define annihilation operator

# Define the Initial State:
xo = 1.00 # Initial Position
po = 0.00 # Initial Momentum

# Define a coherent state with amplitude alpha
psi0 = qt.coherent(N, alpha=(xo+1.j*po)/np.sqrt(2))

# Define the Hamiltonian:
mass = 1 # mass = 1.0
hbar = 1 # hbar = 1.0
omega = 1.0 # Oscillator frequency
H_ho = hbar*omega*(a.dag()*a + 1./2.) # Harmonic Oscillator Hamiltonian

# Define the propagation time array with n_steps from 0 to total_time
n_steps = 200 # Number of time steps
total_time = 10 # Total Propagation time

# Define the list of times for which we calculate dynamics.
tlist = np.linspace(0, total_time, 200)

# Run dynamics!
result = qt.mesolve(H_ho, psi0, tlist, [], [], progress_bar=True,options=qt.solver.Options(nsteps=len(tlist)))

10.1%. Run time:   0.00s. Est. time left: 00:00:00:00
20.1%. Run time:   0.01s. Est. time left: 00:00:00:00
30.2%. Run time:   0.01s. Est. time left: 00:00:00:00
40.2%. Run time:   0.02s. Est. time left: 00:00:00:00
50.3%. Run time:   0.03s. Est. time left: 00:00:00:00
60.3%. Run time:   0.04s. Est. time left: 00:00:00:00
70.4%. Run time:   0.05s. Est. time left: 00:00:00:00
80.4%. Run time:   0.06s. Est. time left: 00:00:00:00
90.5%. Run time:   0.07s. Est. time left: 00:00:00:00
100.0%. Run time:   0.08s. Est. time left: 00:00:00:00
Total run time:   0.08s


/mgpfs/home/mkhairiansyah/.conda/envs/env-ml/lib/python3.10/site-packages/qutip/solver/options.py:16: FutureWarning: Dedicated options class are no longer needed, options should be passed as dict to solvers.
  warnings.warn(
/mgpfs/home/mkhairiansyah/.conda/envs/env-ml/lib/python3.10/site-packages/qutip/solver/solver_base.py:598: FutureWarning: e_ops will be keyword only from qutip 5.3 for all solver
  warnings.warn(
/mgpfs/home/mkhairiansyah/.conda/envs/env-ml/lib/python3.10/site-packages/qutip/solver/solver_base.py:501: FutureWarning: "progress_bar" is now included in options:
 Use `options={"progress_bar": False / True / "tqdm" / "enhanced"}`
  warnings.warn(


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import hermite, factorial

In [4]:
# Operator annihilation
a = qt.destroy(N)

# Operator posisi dan momentum
x_op = (a + a.dag()) / np.sqrt(2)
p_op = -1j * (a - a.dag()) / np.sqrt(2)

# Expectation values
x_exp = qt.expect(x_op, result.states)
p_exp = qt.expect(p_op, result.states)

# Plot
plt.figure()
plt.plot(tlist, x_exp, label='<x>')
plt.plot(tlist, p_exp, label='<p>')
plt.xlabel('Time')
plt.ylabel('Expectation value')
plt.legend()
plt.title('Expectation Values ⟨x⟩ and ⟨p⟩')
plt.show()

In [5]:
# Ambil state terakhir
psi_final = result.states[-1]

# Probabilitas tiap state
prob = np.abs(psi_final.full().flatten())**2

# Plot
plt.figure()
plt.bar(range(len(prob)), prob)
plt.xlabel('Fock state n')
plt.ylabel('Probability |c_n|^2')
plt.title('Fock State Distribution')
plt.show()

In [6]:
def ho_basis(n, x):
    Hn = hermite(n)
    norm = 1.0 / np.sqrt((2**n) * factorial(n) * np.sqrt(np.pi))
    return norm * np.exp(-x**2 / 2) * Hn(x)

In [7]:
x_vals = np.linspace(-5, 5, 200)

In [8]:
def reconstruct_psi(state, x_vals):
    coeffs = state.full().flatten()
    psi_x = np.zeros_like(x_vals, dtype=complex)
    
    for n, c in enumerate(coeffs):
        psi_x += c * ho_basis(n, x_vals)
        
    return psi_x

In [15]:
import matplotlib.pyplot as plt
plt.figure()

for idx in [0, len(tlist)//4, len(tlist)//2, -1]:
    psi_x = reconstruct_psi(result.states[idx], x_vals)
    plt.plot(x_vals, np.abs(psi_x)**2, label=f't={tlist[idx]:.2f}')

plt.xlabel('x')
plt.ylabel('|ψ(x)|²')
plt.title('Wavepacket Evolution (QuTiP)')
plt.legend()
plt.show()
plt.savefig('Harmonic_Oscillator_Dynamics_with_QuTiP.png')

In [16]:
import matplotlib
matplotlib.use('Agg')  # penting untuk HPC

import matplotlib.pyplot as plt
import matplotlib.animation as animation

fig, ax = plt.subplots()
line, = ax.plot([], [])
ax.set_xlabel('x')
ax.set_ylabel('|ψ(x)|²')
ax.set_title('Wavepacket Evolution (QuTiP)')
ax.set_xlim(-5, 5)
ax.set_ylim(0, 1)

def update(frame):
    psi_x = reconstruct_psi(result.states[frame], x_vals)
    line.set_data(x_vals, np.abs(psi_x)**2)
    return line,

ani = animation.FuncAnimation(fig, update, frames=len(tlist), interval=50)

# SAVE sebagai GIF
ani.save("wavepacket.gif", writer='pillow', fps=20)

print("Animasi disimpan sebagai wavepacket.gif")

Animasi disimpan sebagai wavepacket.gif
